This notebook collects all latest data about carceral facilities in California

In [145]:
import pandas as pd
import numpy as np
import geopandas as gpd

## Base facilities data set from FEMA RAPT

In [166]:
fema_boundaries = gpd.read_file("data_sources/facilities/Prison_Boundaries_RAPT.geojson")
len(fema_boundaries)

6471

In [167]:
fema_boundaries = fema_boundaries[fema_boundaries["STATE"] == "CA"]
fema_boundaries = fema_boundaries[fema_boundaries["STATUS"] == "OPEN"].reset_index(drop=True)
len(fema_boundaries)

357

In [168]:
fema_cols_drop = [
    'FID',
    'OBJECTID',
    'NAICS_CODE',
    'NAICS_DESC',
    'SOURCE',
    'SOURCEDATE',
    'VAL_METHOD',
    'VAL_DATE',
    'ZIP4'
]
fema_boundaries = fema_boundaries.drop(columns=fema_cols_drop)

In [169]:
name_cols = ['NAME', 'ADDRESS', 'CITY']
for col in name_cols:
    fema_boundaries[col] = fema_boundaries[col].str.title()
    
# Update all column names to be lowercase
fema_boundaries.columns = [col.lower() for col in fema_boundaries.columns]

## Add centerpoints

In [150]:
def add_facility_centroids(gdf):
    """
    Calculates the centroid of each facility geometry and adds 
    latitude and longitude columns.
    """
    original_crs = gdf.crs
    
    if original_crs is not None:
        centroids = gdf.to_crs(epsg=3857).geometry.centroid.to_crs(original_crs)
    else:
        centroids = gdf.geometry.centroid
    
    # Extract coordinates
    gdf['longitude'] = centroids.x
    gdf['latitude'] = centroids.y
    
    return gdf

In [151]:
facs_with_centers = add_facility_centroids(fema_boundaries)

## Merge with Census Tract ID

In [152]:
tracts = gpd.read_file("data_sources/facilities/cb_2020_06_tract_500k.zip")

In [153]:
def add_census_tracts(gdf, tracts):
    """
    Performs a spatial join using the facility centroids to identify the 
    Census Tract GEOID for each facility.
    """

    points_gdf = gpd.GeoDataFrame(
        gdf[['facilityid']], 
        geometry=gpd.points_from_xy(gdf.longitude, gdf.latitude),
        crs=gdf.crs
    )

    if points_gdf.crs != tracts.crs:
        tracts = tracts.to_crs(points_gdf.crs)

    joined = gpd.sjoin(points_gdf, tracts[['GEOID', 'geometry']], how='left', predicate='within')

    gdf['tract_geoid'] = gdf['facilityid'].map(joined.set_index('facilityid')['GEOID'])

    return gdf

In [154]:
facs_with_centers = add_census_tracts(facs_with_centers, tracts)

## Calculate capacity % where available

In [155]:
def calculate_capacity_metrics(gdf):
    """
    Cleans -999 values and calculates capacity percentage metrics.
    """
    cols_to_clean = ['population', 'capacity']
    for col in cols_to_clean:
        if col in gdf.columns:
            gdf[col] = gdf[col].replace(-999, np.nan)

    # 2. Calculate current capacity_percent (population / capacity)
    if 'population' in gdf.columns and 'capacity' in gdf.columns:
        gdf['capacity_percent'] = gdf['population'] / gdf['capacity']

    return gdf

In [156]:
facs_with_pop = calculate_capacity_metrics(facs_with_centers)

## Re-type after manual review

In [157]:
# After reviewing the FEMA data set, several facilities had their 'type' manually fixed
manual_type_dict = {
    '10000850':'FEDERAL',
    '10000851':'FEDERAL',
    '10000849':'FEDERAL',
    '10000845':'STATE',
    '10000846':'STATE'
}
facs_with_pop['type'] = facs_with_pop['facilityid'].map(manual_type_dict).fillna(facs_with_pop['type'])

## Export

In [158]:
facs_with_pop.to_csv("data/ca_facilities.csv", index=False)

In [159]:
cdcr_facilities = facs_with_pop[facs_with_pop["type"] == "STATE"]

In [162]:
cdcr_facilities.to_csv("data/cdcr_facilities_base.csv", index=False)

In [161]:
len(facs_with_pop), len(cdcr_facilities)

(357, 84)